Understanding the dataset, searching for missing and empty values.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from gridcast.data import load_raw

df = load_raw("../data/raw/data.csv")   
df.info()

In [ ]:
# 1. Time coverage and resolution
print(df.index.min(), "->", df.index.max())
print(df.index.to_series().diff().value_counts().head())

# 2. Missing timestamps
full_idx = pd.date_range(df.index.min(), df.index.max(), freq="15min")
print("expected rows:", len(full_idx), "| actual rows:", len(df),
      "| missing:", len(full_idx) - len(df))

# 3. Nulls and basic stats
print(df.isna().sum()[lambda s: s > 0])
df[["demand_met_mw", "frequency_hz", "solar_mw"]].describe()

Preaparation of Training and Test dataset

In [ ]:
from gridcast.config import load_config
from gridcast.data import load_raw, make_regular
from gridcast.features import build_features

cfg = load_config("../configs/config.yaml")

raw = load_raw("../" + cfg["data"]["raw_path"])
df = make_regular(raw, cfg["data"]["freq"], cfg["data"]["max_gap_steps"])
X, y = build_features(df, cfg)

print(X.shape, y.shape)
print(X.columns.tolist())

X.assign(y=y).to_parquet("../data/processed/features_h96.parquet")

In [ ]:
from gridcast.backtest import run_backtest
from gridcast.models import SeasonalNaive, Blend

X = pd.read_parquet("../data/processed/features_h96.parquet")
y = X.pop("y")

t = cfg["forecast"]["target"]
models = {
    "naive_1d": SeasonalNaive(f"{t}_lag96"),
    "naive_1w": SeasonalNaive(f"{t}_lag672"),
    "blend_1d_1w": Blend([f"{t}_lag96", f"{t}_lag672"]),
}

results = {}
for name, m in models.items():
    res, _ = run_backtest(m, X, y, cfg)
    results[name] = res
    print(name); print(res.round(2).to_string(index=False)); print()

summary = pd.DataFrame({n: r.drop(columns=["fold", "test_start", "n_test"]).mean()
                        for n, r in results.items()}).T
print(summary.round(2))

Results: 

Blended model performs better than the naive 1 day model and 1 week model which just simple predict the value is the previous day/week values. The blending of both of these cancel out for some errors during peak hours. Otherwise, naive 1 day model outperforms the other two models and will be our baseline.
| Model | n_eval | MAE | RMSE | MAPE | MASE | Peak MAPE |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| `naive_1d` | 2688.0 | 5463.94 | 7399.85 | 2.58 | 1.06 | 2.61 |
| `naive_1w` | 2688.0 | 9398.45 | 11809.74 | 4.37 | 1.81 | 3.55 |
| `blend_1d_1w` | 2688.0 | 6001.54 | 7673.19 | 2.81 | 1.16 | 2.50 |

The top 10 worst performing days:

In [ ]:
target = cfg["forecast"]["target"]
na = df[target].isna()
print("rows in df:", len(df), "| rows in X:", len(X), "| dropped:", len(df) - len(X))
print("short gaps repaired:", int(df["was_missing"].sum() - na.sum()))

run_id = (na != na.shift()).cumsum()
gaps = df.index.to_series().groupby(run_id).agg(start="first", n="size")[na.groupby(run_id).first()]
gaps["hours"] = gaps["n"] / 4
print(gaps.sort_values("n", ascending=False).head(10))

# other columns can also cause drops
print(df[[target] + cfg["features"]["gen_cols"]].isna().sum())

In [ ]:
res, preds = run_backtest(models["blend_1d_1w"], X, y, cfg)
preds["ape"] = (preds["y"] - preds["pred"]).abs() / preds["y"] * 100
daily = preds["ape"].resample("D").mean().dropna()

print(daily.sort_values(ascending=False).head(10).round(2))
daily.plot(figsize=(12, 3));

In [ ]:
X, y = build_features(df, cfg)
print("df:", len(df), "| X:", len(X), "| dropped:", len(df) - len(X))   # expect ~1,376
print(X.columns.tolist())
print(X[["is_holiday", "days_to_holiday", "days_since_holiday"]].describe().round(2))
print(X.isna().mean().round(3)[lambda s: s > 0])    # NaN share, only gap-caused now

In [ ]:
# Rebuild features
X, y = build_features(df, cfg)

# Row counts
print("Raw/regularized rows:", len(df))
print("Feature rows:", len(X))
print("Target rows:", len(y))
print("Rows dropped:", len(df) - len(X))

# Date range
print("\nFeature date range:")
print(X.index.min(), "->", X.index.max())

# Feature columns
print("\nFeature columns:")
print(X.columns.tolist())

# Holiday feature statistics
print("\nHoliday features:")
print(
    X[["is_holiday", "days_to_holiday", "days_since_holiday"]]
    .describe()
    .round(2)
)

# Missing feature values
print("\nFeatures with missing values:")
print(X.isna().mean().round(3)[lambda s: s > 0])

# Target integrity
print("\nTarget missing:", y.isna().sum())
print("Target name:", y.name)
print("Index aligned:", X.index.equals(y.index))

In [ ]:
# Replace results with the variable holding your backtest output.
print(res)

# If results is a DataFrame:
print("\nNumber of folds:")
print(res["fold"].nunique())

print("\nRows per fold:")
print(res.groupby("fold").size())